# 🦜 Introduction to LangChain — Hands-On Examples

> **Module 03 | Submodule 01 | Practical Notebook**
>
> This notebook covers:
> - Setting up LangChain
> - Your first LangChain chain
> - Exploring all major components
> - A mini RAG pipeline
> - LangSmith tracing

## 📦 Installation & Setup

In [ ]:
# Install required packages
!pip install langchain langchain-openai langchain-anthropic \
             langchain-community langchain-core \
             langchain-text-splitters \
             python-dotenv chromadb faiss-cpu

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # Load .env file

# Optional: Enable LangSmith tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "module-03-intro"

print("Setup complete! API keys loaded:", bool(os.getenv("OPENAI_API_KEY")))

---
## 1️⃣ Your First LangChain Chain

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Components
llm    = ChatOpenAI(model="gpt-4o-mini", temperature=0)
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that explains concepts clearly."),
    ("human",  "Explain {concept} in 2-3 sentences.")
])
parser = StrOutputParser()

# 2. Chain with LCEL pipe operator
chain = prompt | llm | parser

# 3. Run it
result = chain.invoke({"concept": "LangChain"})
print(result)

In [ ]:
# The chain is a Runnable — try all methods

# invoke — single call
result = chain.invoke({"concept": "LCEL"})
print("invoke:", result[:80], "...")

# stream — token by token
print("\nstream:")
for chunk in chain.stream({"concept": "vector stores"}):
    print(chunk, end="", flush=True)
print()

# batch — multiple inputs in parallel
results = chain.batch([
    {"concept": "embeddings"},
    {"concept": "RAG"},
    {"concept": "agents"}
])
print("\nbatch (3 results):")
for r in results:
    print(f"  - {r[:60]}...")

---
## 2️⃣ Models — LLM vs ChatModel

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Method 1: Using message objects
messages = [
    SystemMessage(content="You are a concise assistant."),
    HumanMessage(content="What is LangChain?")
]
response = llm.invoke(messages)

print("Type:", type(response))          # AIMessage
print("Content:", response.content)
print("Tokens:", response.usage_metadata)

In [ ]:
# Method 2: String input (auto-converts to HumanMessage)
response = llm.invoke("What is LCEL?")
print(response.content)

# Method 3: Tuple format (role, content)
response = llm.invoke([("system", "Be brief."), ("human", "What is RAG?")])
print(response.content)

In [ ]:
# Configuration — override at runtime
response = llm.invoke(
    "Tell me a story",
    config={"configurable": {"temperature": 1.0}}  # More creative
)

# Model metadata
print("Model:", llm.model_name)
print("Temperature:", llm.temperature)

---
## 3️⃣ Prompts — Templates & Formatting

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Basic template with variables
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role} who specializes in {domain}."),
    ("human", "{question}")
])

# Format and inspect
formatted = prompt.format_messages(
    role="senior developer",
    domain="Python",
    question="What is a decorator?"
)
for msg in formatted:
    print(f"[{msg.type}]: {msg.content}")

In [ ]:
# MessagesPlaceholder — inject dynamic conversation history
prompt_with_history = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),  # ← dynamic history
    ("human", "{question}")
])

# Simulate conversation history
history = [
    HumanMessage(content="My name is Alice"),
    AIMessage(content="Nice to meet you, Alice!")
]

chain = prompt_with_history | llm | StrOutputParser()
result = chain.invoke({
    "history": history,
    "question": "What is my name?"
})
print(result)  # Should remember "Alice"

---
## 4️⃣ Output Parsers — Structured Output

In [ ]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate

# StrOutputParser — simplest
chain = ChatPromptTemplate.from_template("Say hello to {name}") | llm | StrOutputParser()
print(chain.invoke({"name": "Alice"}))
print(type(chain.invoke({"name": "Alice"})))  # str

In [ ]:
# PydanticOutputParser — enforce typed schema
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List

class TechStack(BaseModel):
    name: str = Field(description="Name of the tech stack or framework")
    language: str = Field(description="Primary programming language")
    use_cases: List[str] = Field(description="Top 3 use cases")
    difficulty: str = Field(description="Beginner / Intermediate / Advanced")

parser = PydanticOutputParser(pydantic_object=TechStack)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract tech stack info.\n{format_instructions}"),
    ("human", "{text}")
]).partial(format_instructions=parser.get_format_instructions())

chain = prompt | llm | parser

result = chain.invoke({"text": "LangChain is a Python framework for building LLM apps, great for RAG, agents, and chatbots."})
print("Parsed result:")
print(f"  Name: {result.name}")
print(f"  Language: {result.language}")
print(f"  Use Cases: {result.use_cases}")
print(f"  Difficulty: {result.difficulty}")
print(f"  Type: {type(result)}")  # TechStack

---
## 5️⃣ Document Loaders — Data Ingestion

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document

# Load a web page
loader = WebBaseLoader("https://python.langchain.com/docs/introduction/")
docs = loader.load()

print(f"Loaded {len(docs)} document(s)")
print(f"Page content length: {len(docs[0].page_content)} chars")
print(f"Metadata: {docs[0].metadata}")
print(f"First 300 chars:\n{docs[0].page_content[:300]}")

In [ ]:
# Create documents manually (useful for testing)
docs = [
    Document(
        page_content="LangChain is a framework for building LLM applications.",
        metadata={"source": "intro", "section": "overview"}
    ),
    Document(
        page_content="LangGraph is used to build stateful multi-actor applications with LLMs.",
        metadata={"source": "intro", "section": "langgraph"}
    ),
    Document(
        page_content="LangSmith is a platform for LLM observability, tracing, and evaluation.",
        metadata={"source": "intro", "section": "langsmith"}
    ),
]
print(f"Created {len(docs)} documents")

---
## 6️⃣ Text Splitters — Chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create sample long document
long_text = """
LangChain is an open-source framework designed to simplify the creation of applications 
using Large Language Models. It acts as a bridge between the LLM and external data sources,
tools, and systems. LangChain provides modular, composable components that you can mix and 
match to build powerful AI applications.

The key components of LangChain include Models, Prompts, Chains, Agents, Memory, Tools,
Document Loaders, Text Splitters, Vector Stores, and Retrievers. Each component is a 
Runnable — they all support the same .invoke(), .stream(), and .batch() interface.

LangChain Expression Language (LCEL) uses the pipe operator | to compose components into
chains. This is the recommended way to build chains in modern LangChain, as it provides
streaming, async, batching, and full LangSmith observability out of the box.
"""

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
)

chunks = splitter.create_documents([long_text])
print(f"Original length: {len(long_text)} chars")
print(f"Chunks created: {len(chunks)}")
print()
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ({len(chunk.page_content)} chars) ---")
    print(chunk.page_content)
    print()

---
## 7️⃣ Vector Stores & Retrievers — Semantic Search

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# Sample knowledge base
docs = [
    Document(page_content="LangChain helps build LLM applications with modular components."),
    Document(page_content="LangGraph creates stateful, graph-based agent workflows."),
    Document(page_content="LangSmith provides observability and tracing for LLM apps."),
    Document(page_content="LangFlow is a visual, low-code builder for LangChain apps."),
    Document(page_content="LangServe deploys LangChain chains as REST API endpoints."),
    Document(page_content="Embeddings convert text into numeric vectors for semantic search."),
    Document(page_content="FAISS is a fast library for vector similarity search."),
    Document(page_content="ChromaDB is an open-source vector database for RAG applications."),
]

# Create embeddings and vector store
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(docs, embeddings)

# Semantic similarity search
query = "How do I monitor and debug LLM calls?"
results = vectorstore.similarity_search(query, k=2)
print(f"Query: {query}")
print("Top matches:")
for r in results:
    print(f"  - {r.page_content}")

In [ ]:
# Retriever interface (Runnable)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
docs = retriever.invoke("stateful agents")
print("Retrieved for 'stateful agents':")
for d in docs:
    print(f"  - {d.page_content}")

---
## 8️⃣ Tools — Functions Agents Can Call

In [ ]:
from langchain_core.tools import tool

@tool
def get_word_count(text: str) -> int:
    """Count the number of words in a text string."""
    return len(text.split())

@tool
def convert_to_uppercase(text: str) -> str:
    """Convert a text string to uppercase."""
    return text.upper()

@tool
def calculate(expression: str) -> str:
    """Evaluate a safe mathematical expression. Example: '2 + 2 * 10'"""
    try:
        # Only allow safe math operations
        allowed = set('0123456789+-*/().% ')
        if all(c in allowed for c in expression):
            return str(eval(expression))
        return "Error: unsafe expression"
    except Exception as e:
        return f"Error: {e}"

# Inspect tool schema
print("Tool name:", calculate.name)
print("Description:", calculate.description)
print("Args schema:", calculate.args)

# Call directly
print("\nDirect call:")
print(calculate.invoke({"expression": "25 * 17 + 8"}))

In [ ]:
# Bind tools to an LLM — it will decide when to call them
tools = [get_word_count, convert_to_uppercase, calculate]
llm_with_tools = llm.bind_tools(tools)

# Ask the LLM a question that requires a tool
response = llm_with_tools.invoke("What is 144 divided by 12?")
print("Response type:", type(response))
print("Tool calls:", response.tool_calls)

---
## 9️⃣ Mini RAG Pipeline — Putting It All Together

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document

# Knowledge base
knowledge = [
    Document(page_content="LangChain is an open-source framework for building LLM applications."),
    Document(page_content="LangGraph adds stateful, graph-based orchestration to LangChain agents."),
    Document(page_content="LangSmith is an observability tool for LLM tracing, cost tracking, and evaluation."),
    Document(page_content="LCEL (LangChain Expression Language) uses the | operator to compose Runnables."),
    Document(page_content="AgentExecutor runs a ReAct loop: Thought → Action → Observation → Final Answer."),
    Document(page_content="Vector stores store document embeddings and support semantic similarity search."),
    Document(page_content="RAG (Retrieval Augmented Generation) retrieves relevant context before LLM generation."),
]

# Build the vector store
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(knowledge, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# RAG prompt
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a LangChain expert. Answer ONLY based on the context below.
If the answer is not in the context, say 'I don't have that information.'

Context:
{context}"""),
    ("human", "{question}")
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Complete RAG chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | ChatOpenAI(model="gpt-4o-mini", temperature=0)
    | StrOutputParser()
)

# Test queries
questions = [
    "What is LangGraph?",
    "How does LCEL work?",
    "What is the capital of France?"  # Not in knowledge base
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {rag_chain.invoke(q)}")
    print()

---
## 🔍 Inspect Chain Structure

In [ ]:
# Every LCEL chain can be inspected
simple_chain = rag_prompt | ChatOpenAI(model="gpt-4o-mini") | StrOutputParser()

print("Input schema:", simple_chain.input_schema.schema())
print("Output schema:", simple_chain.output_schema.schema())

# Visualize the graph
simple_chain.get_graph().print_ascii()

---
## ✅ Summary & Key Takeaways

In this notebook you have:

| # | Concept | What You Built |
|---|---|---|
| 1 | LCEL | First chain with `prompt \| llm \| parser` |
| 2 | Models | invoke, stream, batch on ChatOpenAI |
| 3 | Prompts | ChatPromptTemplate with MessagesPlaceholder |
| 4 | Output Parsers | PydanticOutputParser for typed structured output |
| 5 | Document Loaders | Loaded a web page into Documents |
| 6 | Text Splitters | Chunked a long document with overlap |
| 7 | Vector Stores | FAISS semantic similarity search |
| 8 | Tools | Custom tools + bind_tools to LLM |
| 9 | RAG | Full end-to-end Retrieval Augmented Generation pipeline |

**Next**: [02 — Models & Prompts](../02_models_and_prompts/theory.md) — deep dive into models and prompt engineering.